# **Problem Statement**

## **Business Context**

Workplace safety in hazardous environments like construction sites and industrial plants is crucial to prevent accidents and injuries. One of the most important safety measures is ensuring workers wear safety helmets, which protect against head injuries from falling objects and machinery. Non-compliance with helmet regulations increases the risk of serious injuries or fatalities, making effective monitoring essential, especially in large-scale operations where manual oversight is prone to errors and inefficiency.

To overcome these challenges, SafeGuard Corp plans to develop an automated image analysis system capable of detecting whether workers are wearing safety helmets. This system will improve safety enforcement, ensuring compliance and reducing the risk of head injuries. By automating helmet monitoring, SafeGuard aims to enhance efficiency, scalability, and accuracy, ultimately fostering a safer work environment while minimizing human error in safety oversight.

## **Objective**

As a data scientist at SafeGuard Corp, you are tasked with developing an image classification model that classifies images into one of two categories:
- **With Helmet:** Workers wearing safety helmets.
- **Without Helmet:** Workers not wearing safety helmets.

## **Data Description**

The dataset consists of **631 images**, equally divided into two categories:

- **With Helmet:** 311 images showing workers wearing helmets.
- **Without Helmet:** 320 images showing workers not wearing helmets.

**Dataset Characteristics:**
- **Variations in Conditions:** Images include diverse environments such as construction sites, factories, and industrial settings, with variations in lighting, angles, and worker postures to simulate real-world conditions.
- **Worker Activities:** Workers are depicted in different actions such as standing, using tools, or moving, ensuring robust model learning for various scenarios.

# **Installing and Importing the Necessary Libraries**

In [9]:
# !pip install tensorflow[and-cuda] numpy==1.25.2 -q

In [185]:
# Import Tensorflow and check GPU access.
import tensorflow as tf
from tensorflow.python.client import device_lib
print("Num GPUs Available:", len(tf.config.list_physical_devices('GPU')))
print(tf.__version__)

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print("GPU Available: Yes")
    print("GPU Name: ", device_lib.list_local_devices()[-1].physical_device_desc)
else:
    print("GPU Available: No")

LoadError: ArgumentError: Package tensorflow not found in current path.
- Run `import Pkg; Pkg.add("tensorflow")` to install the tensorflow package.

**Note:**

- After running the above cell, kindly restart the notebook kernel (for Jupyter Notebook) or runtime (for Google Colab) and run all cells sequentially from the next cell.

- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in this notebook.

In [12]:
# Import required libraires
import os
import random
import numpy as np                                                                               
import pandas as pd
import seaborn as sns
import matplotlib.image as mpimg                                                                 
import matplotlib.pyplot as plt                                                                  
import math                                                                                      
import cv2
import time


# Tensorflow modules
import tensorflow as tf
import keras
from tensorflow.keras.preprocessing.image import ImageDataGenerator                              
from tensorflow.keras.models import Sequential                                                   
from tensorflow.keras.layers import Dense,Dropout,Flatten,Conv2D,MaxPooling2D,BatchNormalization 
from tensorflow.keras.optimizers import Adam,SGD                                                 
from tensorflow.keras.models import Model
from keras.applications.vgg16 import VGG16      
from tensorflow.keras.applications.vgg16 import preprocess_input


#Imports functions for evaluating the performance of machine learning models
from sklearn import preprocessing    
from sklearn.model_selection import train_test_split  
from sklearn.metrics import confusion_matrix
from sklearn.metrics import mean_squared_error as mse           
from sklearn.metrics import (confusion_matrix, ConfusionMatrixDisplay, accuracy_score, precision_score, recall_score, f1_score, classification_report, roc_auc_score)

# Ignore warnings
import warnings
warnings.filterwarnings('ignore')

LoadError: ArgumentError: Package os not found in current path.
- Run `import Pkg; Pkg.add("os")` to install the os package.

In [13]:
# Set the seed using keras.utils.set_random_seed. This will set:
# 1) `numpy` seed
# 2) backend random seed
# 3) `python` random seed
tf.keras.utils.set_random_seed(812)

LoadError: UndefVarError: `tf` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [14]:
# measuring how long it takes to run this notebook.
start_time = time.time()

LoadError: type #time has no field time

# **Data Overview**


## Loading the data

In [17]:
images = np.load('images_proj.npy')
labels = pd.read_csv('Labels_proj.csv')

LoadError: ParseError:
[90m# Error @ [0;0m]8;;file://C:/Users/cmurr/PythonWork/UT_Austin/Computer Vision Intro/Project/In[17]#1:19\[90mIn[17]:1:19[0;0m]8;;\
images = np.load('[48;2;120;70;70mimages_proj.npy[0;0m')
[90m#                 └─────────────┘ ── [0;0m[91mcharacter literal contains multiple characters[0;0m

In [18]:
# Check the shape of images and labels
print(images.shape)
print(labels.shape)

LoadError: UndefVarError: `images` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

# **Exploratory Data Analysis**

### Plot random images from each of the classes and print their corresponding labels.

In [21]:
# separates data into classes
helmet_indices = np.where(labels == 1)[0]
no_helmet_indices = np.where(labels == 0)[0]  

# Select one image from each class
helmet_img = images[np.random.choice(helmet_indices)]
no_helmet_img = images[np.random.choice(no_helmet_indices)]

# Plot the images
fig, axes = plt.subplots(1, 2, figsize=(8, 4))

# Display "With Helmet" image
axes[0].imshow(helmet_img)
axes[0].set_title("Worker WITH Helmet")
axes[0].axis('off')

# Display "Without Helmet" image
axes[1].imshow(no_helmet_img)
axes[1].set_title("Worker WITHOUT Helmet")
axes[1].axis('off')

# Show the plots
plt.tight_layout()
plt.show()

LoadError: UndefVarError: `np` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

## Checking for class imbalance


In [23]:
plt.figure(figsize=(6, 4), dpi=150)
sns.countplot(x=labels.iloc[:, 0], palette=['red', 'blue'])
plt.xlabel("Classes", fontsize=14)
plt.ylabel("Number of Images", fontsize=14)
plt.title("Images per Class", fontsize=16)
plt.xticks(ticks=[0, 1], labels=["Without Helmet (0)", "With Helmet (1)"]) ;

LoadError: UndefVarError: `plt` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

This is a a balanced dataset. While the number of images in each class is not exact, they are close. Each class has just over 300 images. 

# **Data Preprocessing**

## Converting images to grayscale

In [27]:
# Convert all images to grayscale to save data size. 
images_gray = [cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) for img in images]

# Display a random sample of grayscale images
num_samples = 5  # number of images to show
sample_indices = random.sample(range(len(images_gray)), num_samples)

plt.figure(figsize=(12, 6))
for i, idx in enumerate(sample_indices):
    plt.subplot(1, num_samples, i + 1)
    plt.imshow(images_gray[idx], cmap='gray')  # use gray colormap
    plt.axis("off")
    plt.title(f"Image {idx}")
plt.show()


LoadError: UndefVarError: `images` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [28]:
# Checking to confirm that the images are now gray scale. Only height and width, no channels.
print(images[0].shape)        # original image
print(images_gray[0].shape)   # converted grayscale image

LoadError: UndefVarError: `images` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

## Split Dataset

In [30]:
X_train, X_temp, y_train, y_temp = train_test_split(np.array(images_gray),labels , test_size=0.24, random_state=42,stratify=labels)

LoadError: UndefVarError: `np` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [31]:
X_val, X_test, y_val, y_test = train_test_split(X_temp,y_temp , test_size=0.5, random_state=42,stratify=y_temp)

LoadError: UndefVarError: `y_temp` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [32]:
images.shape

LoadError: UndefVarError: `images` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [33]:
X_train.shape

LoadError: UndefVarError: `X_train` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [34]:
X_val.shape

LoadError: UndefVarError: `X_val` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [35]:
X_test.shape

LoadError: UndefVarError: `X_test` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

Checking the shape of the various data elements trying to decide on the appropriate split given the small size of the data set. This may be something that will need to be revisited depending on model performance. 

### Data Normalization

In [38]:
# Normalizing the data, between 0 and 1, to ensure there are no large numbers and to improve calculation speed.
X_train_normalized = X_train.astype('float32')/255.0
X_val_normalized = X_val.astype('float32')/255.0
X_test_normalized = X_test.astype('float32')/255.0

LoadError: ParseError:
[90m# Error @ [0;0m]8;;file://C:/Users/cmurr/PythonWork/UT_Austin/Computer Vision Intro/Project/In[38]#2:38\[90mIn[38]:2:38[0;0m]8;;\
# Normalizing the data, between 0 and 1, to ensure there are no large numbers and to improve calculation speed.
X_train_normalized = X_train.astype('[48;2;120;70;70mfloat32[0;0m')/255.0
[90m#                                    └─────┘ ── [0;0m[91mcharacter literal contains multiple characters[0;0m

# **Model Building**

## Utility Functions

In [41]:
# defining a function to compute different metrics to check performance of a classification model built using statsmodels
def model_performance_classification(model, predictors, target):
    """
    Function to compute different metrics to check classification model performance

    model: classifier
    predictors: independent variables
    target: dependent variable
    """

    # checking which probabilities are greater than threshold
    pred = model.predict(predictors).reshape(-1)>0.5

    target = target.to_numpy().reshape(-1)


    acc = accuracy_score(target, pred)  # to compute Accuracy
    recall = recall_score(target, pred, average='weighted')  # to compute Recall
    precision = precision_score(target, pred, average='weighted')  # to compute Precision
    f1 = f1_score(target, pred, average='weighted')  # to compute F1-score

    # creating a dataframe of metrics
    df_perf = pd.DataFrame({"Accuracy": acc, "Recall": recall, "Precision": precision, "F1 Score": f1,},index=[0],)

    return df_perf

LoadError: UndefVarError: `def` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [42]:
def plot_confusion_matrix(model,predictors,target,ml=False):
    """
    Function to plot the confusion matrix

    model: classifier
    predictors: independent variables
    target: dependent variable
    ml: To specify if the model used is an sklearn ML model or not (True means ML model)
    """

    # checking which probabilities are greater than threshold
    pred = model.predict(predictors).reshape(-1)>0.5

    target = target.to_numpy().reshape(-1)

    # Plotting the Confusion Matrix using confusion matrix() function which is also predefined tensorflow module
    confusion_matrix = tf.math.confusion_matrix(target,pred)
    f, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(
        confusion_matrix,
        annot=True,
        linewidths=.4,
        fmt="d",
        square=True,
        ax=ax
    )
    plt.show()

LoadError: UndefVarError: `def` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

## Model 1: Simple Convolutional Neural Network (CNN)

In [44]:
# Initializing Model
model_1 = Sequential()

# Feature extraction = Convolutional and pooling layers
model_1.add(Conv2D(32, (3, 3), activation='relu', padding="same", input_shape=(200,200,1))) 
model_1.add(MaxPooling2D((4, 4), padding='same'))
model_1.add(Conv2D(16, (3, 3), activation='relu', padding="same")) 
model_1.add(MaxPooling2D((4,4), padding='same')) 
model_1.add(Conv2D(8, (3,3), activation='relu', padding="same")) 
model_1.add(MaxPooling2D((4,4), padding='same')) 

# Flatten and Dense layers
model_1.add(Flatten())
model_1.add(Dense(8, activation='relu'))
model_1.add(Dense(4, activation='relu'))
model_1.add(Dense(1, activation='sigmoid'))  #Using the Sigmoid activation function sine this is binary classification 

# Compile the Model with Adam Optimizer
model_1.compile(optimizer='Adam', loss='binary_crossentropy', metrics=["accuracy", "Precision"]) 

# Summary of the model parameters
model_1.summary()

LoadError: UndefVarError: `Sequential` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [45]:
# To determine input size
images_gray[0].shape

LoadError: UndefVarError: `images_gray` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [46]:
history_1 = model_1.fit(
            X_train_normalized, y_train,
            epochs=15, 
            validation_data=(X_val_normalized,y_val),
            shuffle=True,
            batch_size=20, 
            verbose=2
)

LoadError: UndefVarError: `X_val_normalized` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

### Vizualizing the predictions

In [48]:
plt.plot(history_1.history['accuracy']) 
plt.plot(history_1.history['val_accuracy']) 
plt.title('Model 1 Accuracy per Epoch') 
plt.ylabel('Accuracy') 
plt.xlabel('Epoch') 
plt.legend(['Train', 'Validation'], loc='lower right')
plt.show()

LoadError: ParseError:
[90m# Error @ [0;0m]8;;file://C:/Users/cmurr/PythonWork/UT_Austin/Computer Vision Intro/Project/In[48]#1:29\[90mIn[48]:1:29[0;0m]8;;\
plt.plot(history_1.history['[48;2;120;70;70maccuracy[0;0m']) 
[90m#                           └──────┘ ── [0;0m[91mcharacter literal contains multiple characters[0;0m

In [49]:
model_1_train_perf = model_performance_classification(model_1, X_train_normalized,y_train)

print("Train performance metrics")
print(model_1_train_perf)

LoadError: UndefVarError: `model_performance_classification` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [50]:
model_1_valid_perf = model_performance_classification(model_1, X_val_normalized,y_val)

print("Validation performance metrics")
print(model_1_valid_perf)

LoadError: UndefVarError: `model_performance_classification` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [51]:

y_prob = model_1.predict(X_val_normalized, verbose=0).ravel()   # shape (n,)
y_true = np.asarray(y_val).ravel()
y_pred = (y_prob >= 0.5).astype(int)

# Create Confusion matrix
cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=[0, 1])
disp.plot(cmap=plt.cm.Blues)
plt.title("Confusion Matrix (Validation)")
plt.show()

# 5) Metrics
print("Accuracy :", accuracy_score(y_true, y_pred))
print("Precision:", precision_score(y_true, y_pred, zero_division=0))
print("Recall   :", recall_score(y_true, y_pred, zero_division=0))
print("F1-score :", f1_score(y_true, y_pred, zero_division=0))
print("\nClassification Report:\n", classification_report(y_true, y_pred, digits=4))

# Optional: ROC AUC (works with probabilities)
try:
    print("ROC AUC  :", roc_auc_score(y_true, y_prob))
except ValueError:
    pass


LoadError: UndefVarError: `model_1` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

The standard CNN worked extremely well even with a limited number images. Did it perform too well will need to check against test data to verify. 

## Model 2: (VGG-16 (Base))

In [54]:
# Importing and using the convolution, feature extraction layer of th VGG-16 model.
vgg_model = VGG16(weights='imagenet',include_top=False,input_shape=(200, 200, 3))
vgg_model.summary()

LoadError: ParseError:
[90m# Error @ [0;0m]8;;file://C:/Users/cmurr/PythonWork/UT_Austin/Computer Vision Intro/Project/In[54]#2:28\[90mIn[54]:2:28[0;0m]8;;\
# Importing and using the convolution, feature extraction layer of th VGG-16 model.
vgg_model = VGG16(weights='[48;2;120;70;70mimagenet[0;0m',include_top=False,input_shape=(200, 200, 3))
[90m#                          └──────┘ ── [0;0m[91mcharacter literal contains multiple characters[0;0m

In [55]:
# Freeze VGG-16 layers so they don't pick up training
for layer in vgg_model.layers:
    layer.trainable = False

LoadError: ParseError:
[90m# Error @ [0;0m]8;;file://C:/Users/cmurr/PythonWork/UT_Austin/Computer Vision Intro/Project/In[55]#2:31\[90mIn[55]:2:31[0;0m]8;;\
# Freeze VGG-16 layers so they don't pick up training
[90m#                             ┌[0;0m
for layer in vgg_model.layers:[48;2;120;70;70m[0;0m
[48;2;120;70;70m    [0;0mlayer.trainable = False
[90m#──┘ ── [0;0m[91mline break after `:` in range expression[0;0m

### Create Model 2

In [57]:
model_2 = Sequential()
model_2.add(vgg_model)
model_2.add(Flatten())
model_2.add(Dense(1, activation='sigmoid'))
model_2.compile(optimizer='Adam', loss=keras.losses.BinaryCrossentropy(), metrics=["accuracy", "Precision"])
model_2.summary()

LoadError: UndefVarError: `Sequential` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [58]:
train_datagen = ImageDataGenerator(preprocessing_function=lambda x: preprocess_input(x * 255.0))

LoadError: ParseError:
[90m# Error @ [0;0m]8;;file://C:/Users/cmurr/PythonWork/UT_Austin/Computer Vision Intro/Project/In[58]#1:66\[90mIn[58]:1:66[0;0m]8;;\
train_datagen = ImageDataGenerator(preprocessing_function=lambda [48;2;120;70;70mx: preprocess_input(x * 255.0[0;0m))
[90m#                                                                └───────────────────────────┘ ── [0;0m[91mExpected `)`[0;0m

In [59]:
# Fitting the model
batch_size = 20

# Make 3-channel inputs (N, H, W, 3)
X_train_3ch = np.repeat(X_train_normalized[..., None].astype('float32'), 3, axis=-1)
X_val_3ch   = np.repeat(X_val_normalized[..., None].astype('float32'),   3, axis=-1)

history_2 = model_2.fit(
    train_datagen.flow(X_train_3ch, y_train, batch_size=batch_size, seed=42, shuffle=True),
    epochs=25,
    steps_per_epoch=math.ceil(X_train_3ch.shape[0] / batch_size),  # or omit; Keras can infer
    validation_data=(X_val_3ch, y_val),
    verbose=2
)


LoadError: ParseError:
[90m# Error @ [0;0m]8;;file://C:/Users/cmurr/PythonWork/UT_Austin/Computer Vision Intro/Project/In[59]#5:44\[90mIn[59]:5:44[0;0m]8;;\
# Make 3-channel inputs (N, H, W, 3)
X_train_3ch = np.repeat(X_train_normalized[[48;2;120;70;70m...[0;0m, None].astype('float32'), 3, axis=-1)
[90m#                                          └─┘ ── [0;0m[91minvalid identifier[0;0m

### Visualizing the prediction:

In [61]:
plt.plot(history_2.history['accuracy']) #Complete the code to plot the train metrics
plt.plot(history_2.history['val_accuracy']) #Complete the code to plot the validation data metrics
plt.title('Model 2 Accuracy per Epoch') #Complete the code to define the title for the plot
plt.ylabel('Accuracy') #Complete the code to define the label for the y-axis
plt.xlabel('Epoch') #Complete the code to define the label for the x-axis
plt.legend(['Train', 'Validation'], loc='upper left')
plt.show()

LoadError: ParseError:
[90m# Error @ [0;0m]8;;file://C:/Users/cmurr/PythonWork/UT_Austin/Computer Vision Intro/Project/In[61]#1:29\[90mIn[61]:1:29[0;0m]8;;\
plt.plot(history_2.history['[48;2;120;70;70maccuracy[0;0m']) #Complete the code to plot the train metrics
[90m#                           └──────┘ ── [0;0m[91mcharacter literal contains multiple characters[0;0m

In [62]:
model_2_train_perf = model_performance_classification(model_2, X_train_3ch,y_train)

print("Train performance metrics")
print(model_2_train_perf)

LoadError: UndefVarError: `model_performance_classification` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [63]:
model_2_valid_perf = model_performance_classification(model_2, X_val_3ch,y_val)

print("Validation performance metrics")
print(model_2_valid_perf)

LoadError: UndefVarError: `model_performance_classification` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [64]:

y_prob = model_2.predict(X_val_3ch, verbose=0).ravel()   # shape (n,)
y_true = np.asarray(y_val).ravel()
y_pred = (y_prob >= 0.5).astype(int)

# Create Confusion matrix
cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=[0, 1])
disp.plot(cmap=plt.cm.Blues)
plt.title("Confusion Matrix (Validation)")
plt.show()

# 5) Metrics
print("Accuracy :", accuracy_score(y_true, y_pred))
print("Precision:", precision_score(y_true, y_pred, zero_division=0))
print("Recall   :", recall_score(y_true, y_pred, zero_division=0))
print("F1-score :", f1_score(y_true, y_pred, zero_division=0))
print("\nClassification Report:\n", classification_report(y_true, y_pred, digits=4))

# Optional: ROC AUC (works with probabilities)
try:
    print("ROC AUC  :", roc_auc_score(y_true, y_prob))
except ValueError:
    pass


LoadError: UndefVarError: `model_2` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

Model 2, VGG-16 Base, did poorly. This is undestandable, just the convoultion layer with no prediction layer just doesn't seem like it is a good iead. 

## Model 3: (VGG-16 (Base + FFNN))

### Create Model 3

In [68]:
model_3 = Sequential()

# Feature Extraction 
model_3.add(vgg_model)
model_3.add(Flatten())

# Prediction
model_3.add(Dense(32,activation='relu')) 
model_3.add(Dropout(rate=0.5))
model_3.add(Dense(16,activation='relu')) 
model_3.add(Dropout(rate=0.5))

# Output Layer
model_3.add(Dense(1, activation='sigmoid')) 

opt = Adam(learning_rate=0.5) 

model_3.compile(optimizer='Adam',loss=keras.losses.BinaryCrossentropy(), metrics=["accuracy", "Precision"])
model_3.summary()

LoadError: UndefVarError: `Sequential` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [69]:
batch_size = 20

# Make 3-channel inputs (N, H, W, 3)
X_train_3ch = np.repeat(X_train_normalized[..., None].astype('float32'), 3, axis=-1)
X_val_3ch   = np.repeat(X_val_normalized[..., None].astype('float32'),   3, axis=-1)

history_3 = model_3.fit(
    train_datagen.flow(X_train_3ch, y_train, batch_size=batch_size, seed=42, shuffle=True),
    epochs=25,
    steps_per_epoch=math.ceil(X_train_3ch.shape[0] / batch_size),  # or omit; Keras can infer
    validation_data=(X_val_3ch, y_val),
    verbose=2
)


LoadError: ParseError:
[90m# Error @ [0;0m]8;;file://C:/Users/cmurr/PythonWork/UT_Austin/Computer Vision Intro/Project/In[69]#4:44\[90mIn[69]:4:44[0;0m]8;;\
# Make 3-channel inputs (N, H, W, 3)
X_train_3ch = np.repeat(X_train_normalized[[48;2;120;70;70m...[0;0m, None].astype('float32'), 3, axis=-1)
[90m#                                          └─┘ ── [0;0m[91minvalid identifier[0;0m

#### Visualizing the predictions

In [71]:
plt.plot(history_3.history['accuracy'])
plt.plot(history_3.history['val_accuracy'])
plt.title('Model 3 Accuracy per Epoch')
plt.ylabel('Accuracy')
plt.xlabel('Epoch') 
plt.legend(['Train', 'Validation'], loc='upper left')
plt.show()

LoadError: ParseError:
[90m# Error @ [0;0m]8;;file://C:/Users/cmurr/PythonWork/UT_Austin/Computer Vision Intro/Project/In[71]#1:29\[90mIn[71]:1:29[0;0m]8;;\
plt.plot(history_3.history['[48;2;120;70;70maccuracy[0;0m'])
[90m#                           └──────┘ ── [0;0m[91mcharacter literal contains multiple characters[0;0m

In [72]:
model_3_train_perf = model_performance_classification(model_3, X_train_3ch,y_train)

print("Train performance metrics")
print(model_3_train_perf)

LoadError: UndefVarError: `model_performance_classification` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [73]:
model_3_valid_perf = model_performance_classification(model_3, X_val_3ch,y_val)

print("Validation performance metrics")
print(model_3_valid_perf)

LoadError: UndefVarError: `model_performance_classification` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [74]:

y_prob = model_3.predict(X_val_3ch, verbose=0).ravel()   # shape (n,)
y_true = np.asarray(y_val).ravel()
y_pred = (y_prob >= 0.5).astype(int)

# Create Confusion matrix
cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=[0, 1])
disp.plot(cmap=plt.cm.Blues)
plt.title("Confusion Matrix (Validation)")
plt.show()

# 5) Metrics
print("Accuracy :", accuracy_score(y_true, y_pred))
print("Precision:", precision_score(y_true, y_pred, zero_division=0))
print("Recall   :", recall_score(y_true, y_pred, zero_division=0))
print("F1-score :", f1_score(y_true, y_pred, zero_division=0))
print("\nClassification Report:\n", classification_report(y_true, y_pred, digits=4))

# Optional: ROC AUC (works with probabilities)
try:
    print("ROC AUC  :", roc_auc_score(y_true, y_prob))
except ValueError:
    pass


LoadError: UndefVarError: `model_3` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [75]:
model_3_train_perf = model_performance_classification(model_3, X_train_3ch,y_train)

print("Train performance metrics")
print(model_3_train_perf)

LoadError: UndefVarError: `model_performance_classification` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

Model 3, VGG-16 Base to FFNN, might have actually done worse that than VGG-16 Base. Perhaps not enough immages for the CNN to learn the dataset? Regardless the standard CNN still significatly out performed this model.

## Model 4: (VGG-16 (Base + FFNN + Data Augmentation)

- In most of the real-world case studies, it is challenging to acquire a large number of images and then train CNNs.
- To overcome this problem, one approach we might consider is **Data Augmentation**.
- CNNs have the property of **translational invariance**, which means they can recognise an object even if its appearance shifts translationally in some way. - Taking this attribute into account, we can augment the images using the techniques listed below

    -  Horizontal Flip (should be set to True/False)
    -  Vertical Flip (should be set to True/False)
    -  Height Shift (should be between 0 and 1)
    -  Width Shift (should be between 0 and 1)
    -  Rotation (should be between 0 and 180)
    -  Shear (should be between 0 and 1)
    -  Zoom (should be between 0 and 1) etc.

Remember, **data augmentation should not be used in the validation/test data set**.

In [79]:
model_4 = Sequential()

# Feature Extraction 
model_4.add(vgg_model)
model_4.add(Flatten())

# Prediction
model_4.add(Dense(32,activation='relu')) 
model_4.add(Dropout(rate=0.5))
model_4.add(Dense(16,activation='relu')) 
model_4.add(Dropout(rate=0.5))

# Output Layer
model_4.add(Dense(1, activation='sigmoid')) 

opt = Adam(learning_rate=0.5) 

model_4.compile(optimizer='Adam',loss=keras.losses.BinaryCrossentropy(), metrics=["accuracy", "Precision"])
model_4.summary()

LoadError: UndefVarError: `Sequential` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [80]:
train_datagen = ImageDataGenerator(
                              rotation_range=0.5, 
                              fill_mode='nearest',width_shift_range=0.5,height_shift_range=0.5,shear_range=0.5,zoom_range=0.5
                              )

LoadError: ParseError:
[90m# Error @ [0;0m]8;;file://C:/Users/cmurr/PythonWork/UT_Austin/Computer Vision Intro/Project/In[80]#3:42\[90mIn[80]:3:42[0;0m]8;;\
                              rotation_range=0.5, 
                              fill_mode='[48;2;120;70;70mnearest[0;0m',width_shift_range=0.5,height_shift_range=0.5,shear_range=0.5,zoom_range=0.5
[90m#                                        └─────┘ ── [0;0m[91mcharacter literal contains multiple characters[0;0m

In [81]:
batch_size = 20

# Make 3-channel inputs (N, H, W, 3)
X_train_3ch = np.repeat(X_train_normalized[..., None].astype('float32'), 3, axis=-1)
X_val_3ch   = np.repeat(X_val_normalized[..., None].astype('float32'),   3, axis=-1)

history_4 = model_4.fit(
    train_datagen.flow(X_train_3ch, y_train, batch_size=batch_size, seed=42, shuffle=True),
    epochs=25,
    steps_per_epoch=math.ceil(X_train_3ch.shape[0] / batch_size),  # or omit; Keras can infer
    validation_data=(X_val_3ch, y_val),
    verbose=2
)

LoadError: ParseError:
[90m# Error @ [0;0m]8;;file://C:/Users/cmurr/PythonWork/UT_Austin/Computer Vision Intro/Project/In[81]#4:44\[90mIn[81]:4:44[0;0m]8;;\
# Make 3-channel inputs (N, H, W, 3)
X_train_3ch = np.repeat(X_train_normalized[[48;2;120;70;70m...[0;0m, None].astype('float32'), 3, axis=-1)
[90m#                                          └─┘ ── [0;0m[91minvalid identifier[0;0m

#### Visualizing the predictions

In [83]:
plt.plot(history_4.history['accuracy'])
plt.plot(history_4.history['val_accuracy'])
plt.title('Model 4 Accuracy per Epoch')
plt.ylabel('Accuracy')
plt.xlabel('Epoch') 
plt.legend(['Train', 'Validation'], loc='upper left')
plt.show()

LoadError: ParseError:
[90m# Error @ [0;0m]8;;file://C:/Users/cmurr/PythonWork/UT_Austin/Computer Vision Intro/Project/In[83]#1:29\[90mIn[83]:1:29[0;0m]8;;\
plt.plot(history_4.history['[48;2;120;70;70maccuracy[0;0m'])
[90m#                           └──────┘ ── [0;0m[91mcharacter literal contains multiple characters[0;0m

In [84]:
model_4_train_perf = model_performance_classification(model_4, X_train_3ch,y_train)

print("Train performance metrics")
print(model_4_train_perf)

LoadError: UndefVarError: `model_performance_classification` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [85]:
model_4_valid_perf = model_performance_classification(model_4, X_val_3ch,y_val)

print("Validation performance metrics")
print(model_4_valid_perf)

LoadError: UndefVarError: `model_performance_classification` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [86]:

y_prob = model_4.predict(X_val_3ch, verbose=0).ravel()   # shape (n,)
y_true = np.asarray(y_val).ravel()
y_pred = (y_prob >= 0.5).astype(int)

# Create Confusion matrix
cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=[0, 1])
disp.plot(cmap=plt.cm.Blues)
plt.title("Confusion Matrix (Validation)")
plt.show()

# 5) Metrics
print("Accuracy :", accuracy_score(y_true, y_pred))
print("Precision:", precision_score(y_true, y_pred, zero_division=0))
print("Recall   :", recall_score(y_true, y_pred, zero_division=0))
print("F1-score :", f1_score(y_true, y_pred, zero_division=0))
print("\nClassification Report:\n", classification_report(y_true, y_pred, digits=4))

# Optional: ROC AUC (works with probabilities)
try:
    print("ROC AUC  :", roc_auc_score(y_true, y_prob))
except ValueError:
    pass


LoadError: UndefVarError: `model_4` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

# **Model Performance Comparison and Final Model Selection**

In [88]:
# training performance comparison

models_train_comp_df = pd.concat(
    [
        model_1_train_perf.T,
        model_2_train_perf.T,
        model_3_train_perf.T,
        model_4_train_perf.T,
    ],
    axis=1,
)
models_train_comp_df.columns = [
    "Simple Convolutional Neural Network (CNN)","VGG-16 (Base)","VGG-16 (Base+FFNN)","VGG-16 (Base+FFNN+Data Aug)"
]

LoadError: UndefVarError: `model_1_train_perf` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [89]:
models_valid_comp_df = pd.concat(
    [
        model_1_valid_perf.T,
        model_2_valid_perf.T,
        model_3_valid_perf.T,
        model_4_valid_perf.T

    ],
    axis=1,
)
models_valid_comp_df.columns = [
 "Simple Convolutional Neural Network (CNN)","VGG-16 (Base)","VGG-16 (Base+FFNN)","VGG-16 (Base+FFNN+Data Aug)"
]

LoadError: UndefVarError: `model_1_valid_perf` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [90]:
models_train_comp_df

LoadError: UndefVarError: `models_train_comp_df` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [91]:
models_valid_comp_df

LoadError: UndefVarError: `models_valid_comp_df` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [92]:
models_train_comp_df - models_valid_comp_df

LoadError: UndefVarError: `models_train_comp_df` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

## Test Performance - VGG-16(Base + FFNN + Data Augmentation)

In [94]:
X_test_normalized.shape

LoadError: UndefVarError: `X_test_normalized` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [95]:
# X_test_4d = np.expand_dims(X_test_normalized.astype('float32'), axis=-1)
X_test_3ch = np.repeat(X_test_normalized[..., None].astype('float32'), 3, axis=-1)

LoadError: ParseError:
[90m# Error @ [0;0m]8;;file://C:/Users/cmurr/PythonWork/UT_Austin/Computer Vision Intro/Project/In[95]#2:42\[90mIn[95]:2:42[0;0m]8;;\
# X_test_4d = np.expand_dims(X_test_normalized.astype('float32'), axis=-1)
X_test_3ch = np.repeat(X_test_normalized[[48;2;120;70;70m...[0;0m, None].astype('float32'), 3, axis=-1)
[90m#                                        └─┘ ── [0;0m[91minvalid identifier[0;0m

In [96]:
model_test_perf = model_performance_classification(model_4, X_test_3ch,y_test)

LoadError: UndefVarError: `model_performance_classification` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [97]:
model_test_perf

LoadError: UndefVarError: `model_test_perf` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [98]:
plot_confusion_matrix(model_4, X_test_3ch,y_test) 

LoadError: UndefVarError: `plot_confusion_matrix` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

## Test Performance - Simple CNN

In [100]:
model_test_perf = model_performance_classification(model_1, X_test_normalized,y_test)

LoadError: UndefVarError: `model_performance_classification` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [101]:
model_test_perf

LoadError: UndefVarError: `model_test_perf` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [102]:
plot_confusion_matrix(model_1, X_test_normalized,y_test)

LoadError: UndefVarError: `plot_confusion_matrix` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

# Model Selection

Of the four models we developed in this notebook the Simple Convolutional Neural Network and the VGG-16 Fully Loaded (Base + FFNN + Data Augmentation) are a tie. I think simplier is better, since performance is equal, and I am going to select the Simple CNN. The simplier model will be easier to develop and deploy. The pipelines supporting it will also be less complex, faster and, less resource intensive. 

Other Models, the VGG-16 Base and VGG-16 Bae + FFNN did wirse than poorly. It is like they could not learn. I under this for the base model that was just a convolution feature extractor. Regardless these two models did not make the cut. 
    

# **Actionable Insights & Recommendations**

- Model selected - simple/standard CNN. Simplier is better. 
- Model deployment 
  - Successfully deployed this model will provide a safer work environment for affected employees
  - It is recommended that a app with a supporting pipeline be developed in Streamlit, Flask, or Django, to take feeds from cameras - fixed or drones, that provide input data to the model to identify employees not wearing their hardhats. This information could then be forawrded to supervisors for corrective action.
  - Depending on the work site the model could be trained to look for other personal protective equipment as required.
  - Continue collecting data and training the model on larger datasets to ensure functionality and relevance are maintained. 

<font size=5 color='blue'>Power Ahead!</font>
___

In [108]:
# Monitoring how long it takes this notebook to run a various platforms. 
end_time = time.time()
total_time = (end_time - start_time)/60
print(round(total_time,3))

LoadError: type #time has no field time